In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss, brier_score_loss, confusion_matrix
from sklearn.model_selection import train_test_split

training_dataset = pd.read_csv("encoded_training_table.csv")

## Chronological Splitting

In [2]:
target = training_dataset['responded']
general_model_dataset = training_dataset.copy()
users_columns = ['prev_notifications', 'prev_response_rate', 'recent_response_rate', 'slot_prev_response_rate', 'mins_since_prev_notification']
general_model_dataset = general_model_dataset.drop(columns= users_columns + ['uuid', 'responded'])

split_index = int(len(training_dataset) * 0.80)

general_model_X_train = general_model_dataset.iloc[:split_index]
general_model_y_train = target.iloc[:split_index]
general_model_X_test = general_model_dataset.iloc[split_index:]
general_model_y_test = target.iloc[split_index:]

# general_model_X_train, general_model_X_test, general_model_y_train, general_model_y_test = train_test_split(
#     general_model_dataset, target, test_size=0.2, random_state=42, stratify=target
# )

In [10]:
target = training_dataset['responded']
personal_model_dataset = training_dataset.copy()
personal_model_dataset = personal_model_dataset.drop(columns= ['uuid','responded'])

split_index = int(len(training_dataset) * 0.80)

personal_model_X_train = personal_model_dataset.iloc[:split_index]
personal_model_y_train = target.iloc[:split_index]
personal_model_X_test = personal_model_dataset.iloc[split_index:]
personal_model_y_test = target.iloc[split_index:]
y_test = training_dataset.iloc[split_index:]

# general_model_X_train, general_model_X_test, general_model_y_train, general_model_y_test = train_test_split(
#     general_model_dataset, target, test_size=0.2, random_state=42, stratify=target
# )

## Data Scaling

In [11]:
scale_cols = [
    'battery_level',
    'light_level',
    'noise_level',
    'acc_avg',
    'notifications_30m',
    'app_events_30m',
    'unlocks_30m',
    'age'
]

scaler = StandardScaler()
general_model_X_train_scaled = general_model_X_train.copy()
general_model_X_test_scaled = general_model_X_test.copy()

general_model_X_train_scaled[scale_cols] = scaler.fit_transform(general_model_X_train[scale_cols])
general_model_X_test_scaled[scale_cols] = scaler.transform(general_model_X_test[scale_cols])

In [12]:
scale_cols2 = [
    'battery_level',
    'light_level',
    'noise_level',
    'acc_avg',
    'notifications_30m',
    'app_events_30m',
    'unlocks_30m',
    'prev_notifications',
    'prev_response_rate',
    'recent_response_rate',
    'slot_prev_response_rate',
    'mins_since_prev_notification',
    'age'
]

scaler = StandardScaler()
personal_model_X_train_scaled = personal_model_X_train.copy()
personal_model_X_test_scaled = personal_model_X_test.copy()

personal_model_X_train_scaled[scale_cols2] = scaler.fit_transform(personal_model_X_train[scale_cols2])
personal_model_X_test_scaled[scale_cols2] = scaler.transform(personal_model_X_test[scale_cols2])

## General Logistic Regression Model Training

In [13]:
general_model = LogisticRegression(max_iter=2000, random_state=42)
general_model.fit(general_model_X_train_scaled, general_model_y_train)

personal_model = LogisticRegression(max_iter=2000, random_state=42)
personal_model.fit(personal_model_X_train_scaled, personal_model_y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,2000
,multi_class,'deprecated'


In [48]:
# General model predictions
y_pred_general = general_model.predict(general_model_X_test_scaled)
y_prob_general = general_model.predict_proba(general_model_X_test_scaled)[:, 1]


# User-specific model predictions
# y_pred_personal = personal_model.predict(personal_model_X_test_scaled)
y_prob_personal = personal_model.predict_proba(personal_model_X_test_scaled)[:, 1]
y_pred_personal = (y_prob_personal >= 0.465).astype(int)

In [49]:
# --------------------------------------------------
# Step 1: Combine test information and predictions
# --------------------------------------------------

# Each row represents one notification from one user.
# All arrays must have the same length and row order.
uuid_test = y_test["uuid"].copy()

results = pd.DataFrame({
    "uuid": np.asarray(uuid_test),
    "actual": np.asarray(general_model_y_test),
    "general_pred": np.asarray(y_pred_general),
    "general_prob": np.asarray(y_prob_general),
    "personal_pred": np.asarray(y_pred_personal),
    "personal_prob": np.asarray(y_prob_personal)
})


# --------------------------------------------------
# Step 2: Function to calculate evaluation metrics
# --------------------------------------------------

def calculate_metrics(actual, predicted, probabilities):

    # Some users might have only positive or only negative
    # observations in their test data.
    # ROC-AUC cannot be calculated in that situation.

    if len(np.unique(actual)) == 2:
        auc = roc_auc_score(actual, probabilities)
    else:
        auc = np.nan

    return {
        "Accuracy": accuracy_score(actual, predicted),

        "Precision": precision_score(
            actual, predicted, zero_division=0
        ),

        "Recall": recall_score(
            actual, predicted, zero_division=0
        ),

        "F1-score": f1_score(
            actual, predicted, zero_division=0
        ),

        "ROC-AUC": auc,

        "Log loss": log_loss(
            actual, probabilities, labels=[0, 1]
        ),

        "Brier score": brier_score_loss(
            actual, probabilities
        )
    }


# --------------------------------------------------
# Step 3: Evaluate each user independently
# --------------------------------------------------

user_results = []

for uuid, user_data in results.groupby("uuid"):

    actual = user_data["actual"]

    # Calculate metrics for the general model
    general_metrics = calculate_metrics(
        actual,
        user_data["general_pred"],
        user_data["general_prob"]
    )

    # Calculate metrics for the user-specific model
    personal_metrics = calculate_metrics(
        actual,
        user_data["personal_pred"],
        user_data["personal_prob"]
    )

    # Store the results for this user
    user_results.append({

        "uuid": uuid,

        # Number of test observations for this user
        "test_samples": len(user_data),

        # Number of actual responses
        "actual_responses": int(actual.sum()),

        # General model metrics
        **{
            f"General_{key}": value
            for key, value in general_metrics.items()
        },

        # User-specific model metrics
        **{
            f"Personal_{key}": value
            for key, value in personal_metrics.items()
        }
    })


# --------------------------------------------------
# Step 4: Create a results table
# --------------------------------------------------

user_metrics = pd.DataFrame(user_results)

# Sort users by their UUID
user_metrics = user_metrics.sort_values("uuid")

# Display all results
print(user_metrics.to_string(index=False))

# # Save results to a CSV file
# user_metrics.to_csv(
#     "user_evaluation_results.csv",
#     index=False
# )


uuid  test_samples  actual_responses  General_Accuracy  General_Precision  General_Recall  General_F1-score  General_ROC-AUC  General_Log loss  General_Brier score  Personal_Accuracy  Personal_Precision  Personal_Recall  Personal_F1-score  Personal_ROC-AUC  Personal_Log loss  Personal_Brier score
u000            13                 2          0.615385           0.200000        0.500000          0.285714         0.636364          0.591479             0.209526           0.692308            0.000000         0.000000           0.000000          0.681818           0.496034              0.169432
u002             7                 2          1.000000           1.000000        1.000000          1.000000         1.000000          0.326132             0.088918           1.000000            1.000000         1.000000           1.000000          1.000000           0.313067              0.092206
u003            94                 2          0.936170           0.166667        0.500000          0.25000

In [50]:

# Metrics where higher values indicate better performance
higher_is_better = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "ROC-AUC"
]

# Metrics where lower values indicate better performance
lower_is_better = [
    "Log loss",
    "Brier score"
]


# Calculate the difference for each metric
for metric in higher_is_better + lower_is_better:

    user_metrics[f"Difference_{metric}"] = (
        user_metrics[f"Personal_{metric}"]
        - user_metrics[f"General_{metric}"]
    )


# Display only the most useful comparison columns
comparison_columns = [
    "uuid",
    "test_samples",
    "Difference_Accuracy",
    "Difference_Precision",
    "Difference_Recall",
    "Difference_F1-score",
    "Difference_ROC-AUC",
    "Difference_Log loss",
    "Difference_Brier score"
]

comparison = user_metrics[comparison_columns]

print(comparison.to_string(index=False))


uuid  test_samples  Difference_Accuracy  Difference_Precision  Difference_Recall  Difference_F1-score  Difference_ROC-AUC  Difference_Log loss  Difference_Brier score
u000            13             0.076923             -0.200000          -0.500000            -0.285714        4.545455e-02            -0.095445               -0.040094
u002             7             0.000000              0.000000           0.000000             0.000000        0.000000e+00            -0.013065                0.003288
u003            94             0.010638              0.033333           0.000000             0.035714        5.434783e-03            -0.034493               -0.010998
u004            54            -0.203704              0.000000           0.000000             0.000000                 NaN             0.174043                0.076379
u005            68             0.000000              0.000000           0.000000             0.000000        4.807692e-03             0.009470                0.00537

In [51]:

# Count how many users have better results with personalization

print("Users benefiting from personalization:")
print("---------------------------------------")

for metric in higher_is_better:

    better = (
        user_metrics[f"Personal_{metric}"]
        > user_metrics[f"General_{metric}"]
    ).sum()

    worse = (
        user_metrics[f"Personal_{metric}"]
        < user_metrics[f"General_{metric}"]
    ).sum()

    equal = (
        user_metrics[f"Personal_{metric}"]
        == user_metrics[f"General_{metric}"]
    ).sum()

    print(f"\n{metric}:")
    print(f"  Personal model better: {better}")
    print(f"  General model better:  {worse}")
    print(f"  Equal:                 {equal}")


for metric in lower_is_better:

    better = (
        user_metrics[f"Personal_{metric}"]
        < user_metrics[f"General_{metric}"]
    ).sum()

    worse = (
        user_metrics[f"Personal_{metric}"]
        > user_metrics[f"General_{metric}"]
    ).sum()

    equal = (
        user_metrics[f"Personal_{metric}"]
        == user_metrics[f"General_{metric}"]
    ).sum()

    print(f"\n{metric}:")
    print(f"  Personal model better: {better}")
    print(f"  General model better:  {worse}")
    print(f"  Equal:                 {equal}")


Users benefiting from personalization:
---------------------------------------

Accuracy:
  Personal model better: 147
  General model better:  65
  Equal:                 68

Precision:
  Personal model better: 115
  General model better:  96
  Equal:                 69

Recall:
  Personal model better: 81
  General model better:  92
  Equal:                 107

F1-score:
  Personal model better: 117
  General model better:  100
  Equal:                 63

ROC-AUC:
  Personal model better: 161
  General model better:  82
  Equal:                 19

Log loss:
  Personal model better: 213
  General model better:  67
  Equal:                 0

Brier score:
  Personal model better: 214
  General model better:  66
  Equal:                 0


In [52]:
# Calculate the average metric value across users.
# Each user contributes equally, regardless of their test sample size.

metric_columns = [
    "General_Accuracy", "Personal_Accuracy",
    "General_Precision", "Personal_Precision",
    "General_Recall", "Personal_Recall",
    "General_F1-score", "Personal_F1-score",
    "General_ROC-AUC", "Personal_ROC-AUC",
    "General_Log loss", "Personal_Log loss",
    "General_Brier score", "Personal_Brier score"
]

average_per_user = user_metrics[metric_columns].mean()

print("Average performance across users:")
print(average_per_user)

Average performance across users:
General_Accuracy        0.780125
Personal_Accuracy       0.789227
General_Precision       0.442514
Personal_Precision      0.456357
General_Recall          0.312652
Personal_Recall         0.323270
General_F1-score        0.330314
Personal_F1-score       0.333169
General_ROC-AUC         0.731835
Personal_ROC-AUC        0.739031
General_Log loss        0.465958
Personal_Log loss       0.442532
General_Brier score     0.152494
Personal_Brier score    0.143571
dtype: float64


In [62]:

import numpy as np
import pandas as pd


# --------------------------------------------------
# Step 1: Prepare the data
# --------------------------------------------------

uuids = np.asarray(uuid_test)
actual = np.asarray(general_model_y_test)
personal_prob = np.asarray(y_prob_personal)
general_pred = np.asarray(y_pred_general)

# Assign an integer index to each user.
# This allows us to calculate metrics for all users
# simultaneously using NumPy.

unique_users, user_indices = np.unique(
    uuids, return_inverse=True
)

n_users = len(unique_users)

# Actual labels as binary values
actual = actual.astype(int)


# --------------------------------------------------
# Step 2: Calculate general model baseline
# --------------------------------------------------

# Calculate confusion counts for each user.

def calculate_macro_metrics(actual, predicted, user_indices, n_users):

    # Encode each observation as belonging to a user.
    # Calculate TP, FP, TN and FN independently.

    tp = np.bincount(
        user_indices,
        weights=((actual == 1) & (predicted == 1)),
        minlength=n_users
    )

    fp = np.bincount(
        user_indices,
        weights=((actual == 0) & (predicted == 1)),
        minlength=n_users
    )

    tn = np.bincount(
        user_indices,
        weights=((actual == 0) & (predicted == 0)),
        minlength=n_users
    )

    fn = np.bincount(
        user_indices,
        weights=((actual == 1) & (predicted == 0)),
        minlength=n_users
    )

    # Calculate metrics for each user
    accuracy = (tp + tn) / (tp + tn + fp + fn)

    precision = np.divide(
        tp, tp + fp,
        out=np.zeros(n_users, dtype=float),
        where=(tp + fp) != 0
    )

    recall = np.divide(
        tp, tp + fn,
        out=np.zeros(n_users, dtype=float),
        where=(tp + fn) != 0
    )

    f1 = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros(n_users, dtype=float),
        where=(precision + recall) != 0
    )

    # Return macro-average metrics across users
    return {
        "accuracy": np.mean(accuracy),
        "precision": np.mean(precision),
        "recall": np.mean(recall),
        "f1": np.mean(f1)
    }


# General model's baseline performance
general_metrics = calculate_macro_metrics(
    actual,
    general_pred,
    user_indices,
    n_users
)

general_accuracy = general_metrics["accuracy"]

print("General model baseline:")
print(general_metrics)


# --------------------------------------------------
# Step 3: Evaluate different personal thresholds
# --------------------------------------------------

thresholds = np.arange(0.30, 0.601, 0.005)

threshold_results = []

for threshold in thresholds:

    # Convert personal probabilities into predictions
    personal_pred = (
        personal_prob >= threshold
    ).astype(int)

    # Calculate average metrics across all users
    metrics = calculate_macro_metrics(
        actual,
        personal_pred,
        user_indices,
        n_users
    )

    # Calculate balanced score
    balanced_score = np.mean([
        metrics["accuracy"],
        metrics["precision"],
        metrics["recall"],
        metrics["f1"]
    ])

    threshold_results.append({
        "threshold": threshold,
        **metrics,
        "balanced_score": balanced_score
    })


# --------------------------------------------------
# Step 4: Find the best balanced threshold
# --------------------------------------------------

threshold_df = pd.DataFrame(threshold_results)

# Require personal accuracy to be at least equal
# to the general model's macro-average accuracy.

eligible = threshold_df[
    threshold_df["accuracy"] >= general_accuracy
]

if not eligible.empty:

    best_result = eligible.loc[
        eligible["balanced_score"].idxmax()
    ]

    print("\nBest balanced threshold:")
    print(best_result)

else:

    print("\nNo threshold satisfies the accuracy condition.")


# --------------------------------------------------
# Step 5: Display the top 10 thresholds
# --------------------------------------------------

print("\nTop 10 balanced thresholds:")

print(
    eligible.sort_values(
        "balanced_score",
        ascending=False
    ).head(10).to_string(index=False)
)

# Save results
threshold_df.to_csv(
    "threshold_comparison.csv",
    index=False
)


General model baseline:
{'accuracy': np.float64(0.7801248590469747), 'precision': np.float64(0.44251353096120716), 'recall': np.float64(0.312651510158571), 'f1': np.float64(0.3303139731960251)}

Best balanced threshold:
threshold         0.430000
accuracy          0.781541
precision         0.446886
recall            0.365797
f1                0.355654
balanced_score    0.487470
Name: 26, dtype: float64

Top 10 balanced thresholds:
 threshold  accuracy  precision   recall       f1  balanced_score
     0.430  0.781541   0.446886 0.365797 0.355654        0.487470
     0.435  0.783520   0.449943 0.360262 0.353358        0.486771
     0.440  0.784817   0.447897 0.354474 0.349947        0.484284
     0.445  0.785879   0.452129 0.347948 0.346360        0.483079
     0.450  0.787064   0.452670 0.342425 0.344003        0.481541
     0.455  0.787576   0.452212 0.336311 0.339986        0.479021
     0.465  0.789227   0.456357 0.323270 0.333169        0.475505
     0.460  0.788485   0.452127 0.32

## General Logistic Regression Model Testing

In [64]:
# y_predictions = general_model.predict(general_model_X_test_scaled)
y_probability = general_model.predict_proba(general_model_X_test_scaled)[:, 1]
y_predictions = (y_probability >= 0.5).astype(int)

print("Accuracy:", round(accuracy_score(general_model_y_test, y_predictions), 3))
print("Precision:", round(precision_score(general_model_y_test, y_predictions, zero_division=0), 3))
print("Recall:", round(recall_score(general_model_y_test, y_predictions, zero_division=0), 3))
print("F1-score:", round(f1_score(general_model_y_test, y_predictions, zero_division=0), 3))
# print("ROC-AUC:", round(roc_auc_score(general_model_y_test, y_probability), 3))
# print("Log loss:", round(log_loss(general_model_y_test, y_probability), 3))
print("Brier score:", round(brier_score_loss(general_model_y_test, y_probability), 3))
print("Confusion Matrix:")
print(confusion_matrix(general_model_y_test, y_predictions))

Accuracy: 0.791
Precision: 0.487
Recall: 0.358
F1-score: 0.413
Brier score: 0.146
Confusion Matrix:
[[16210  1750]
 [ 2979  1662]]


## General Logistic Regression Model Time Slot Predictions

In [67]:
time_slots = [
    "early morning",
    "mid morning",
    "late morning",
    "afternoon",
    "evening",
    "night"
]

results = []

for slot in time_slots:

    # Copy the same user context.
    candidate = general_model_X_test_scaled.copy()

    # Reset all time-slot columns to 0.
    for name in time_slots[1:]:
        candidate[f"time_slot_{name}"] = 0

    # Activate the selected time slot.
    if(slot != "early morning"):
        candidate[f"time_slot_{slot}"] = 1

    # Predict the probability of responding.
    probabilities = general_model.predict_proba(candidate)[:, 1]

    results.append({
        "time_slot": slot,
        "average_response_probability": round(probabilities.mean() * 100, 2)
    })

# Display results from highest to lowest probability.
results = pd.DataFrame(results).sort_values(
    "average_response_probability",
    ascending=False
)

print(results.to_string(index=False))

print(
    "\nRecommended time slot:",
    results.iloc[0]["time_slot"]
)

    time_slot  average_response_probability
 late morning                         30.78
  mid morning                         28.80
      evening                         28.52
    afternoon                         28.26
        night                         20.34
early morning                         11.67

Recommended time slot: late morning


In [64]:
y_train = general_model_y_train
y_test = general_model_y_test

# Baseline 1: predict the majority class
majority_class = y_train.value_counts().idxmax()

majority_predictions = np.full(len(y_test), majority_class)

print("Baseline 1: majority class")
print("Majority class:", majority_class)

print("Accuracy:", accuracy_score(y_test, majority_predictions))


# Baseline 2: Predict the training response rate
training_response_rate = y_train.mean()

constant_probabilities = np.full(len(y_test), training_response_rate)

print("\nBaseline 2: historical response rate")
print("Training response rate:", round(training_response_rate, 4))
print("Brier score:", round(brier_score_loss(y_test, constant_probabilities), 4))
print("Log loss:", round(log_loss(y_test, constant_probabilities), 4))


# Evaluate your Logistic Regression model
y_predictions = general_model.predict(general_model_X_test_scaled)
y_probability = general_model.predict_proba(general_model_X_test_scaled)[:, 1]

print("Accuracy:", accuracy_score(general_model_y_test, y_predictions))
print("Precision:", precision_score(general_model_y_test, y_predictions, zero_division=0))
print("Recall:", recall_score(general_model_y_test, y_predictions, zero_division=0))
print("F1-score:", f1_score(general_model_y_test, y_predictions, zero_division=0))
print("ROC-AUC:", roc_auc_score(general_model_y_test, y_probability))
print("Log loss:", log_loss(general_model_y_test, y_probability))
print("Brier score:", brier_score_loss(general_model_y_test, y_probability))
print("Confusion Matrix:")
print(confusion_matrix(general_model_y_test, y_predictions))

Baseline 1: majority class
Majority class: 0
Accuracy: 0.794655103756471

Baseline 2: historical response rate
Training response rate: 0.2997
Brier score: 0.1721
Log loss: 0.5305
Accuracy: 0.8054510862351223
Precision: 0.5417522245037646
Recall: 0.34109028226675286
F1-score: 0.41861695094539203
ROC-AUC: 0.7839428661647974
Log loss: 0.42672627344042624
Brier score: 0.1372115250255239
Confusion Matrix:
[[16621  1339]
 [ 3058  1583]]


In [66]:

import pandas as pd

time_slots = [
    "early morning",
    "mid morning",
    "late morning",
    "afternoon",
    "evening",
    "night"
]

# Use the original dataset, before scaling.
train_original = training_dataset.iloc[:split_index].copy()
test_original = training_dataset.iloc[split_index:].copy()

# Recover each notification's actual time slot.
# Early morning is the omitted reference category.
def get_time_slot(row):
    for slot in time_slots[1:]:
        column = f"time_slot_{slot}"

        if row[column] == 1:
            return slot

    return "early morning"

train_original["time_slot"] = train_original.apply(
    get_time_slot, axis=1
)

test_original["time_slot"] = test_original.apply(
    get_time_slot, axis=1
)

# 1. Calculate response rates in the training data.
training_slot_rates = (
    train_original.groupby("time_slot")["responded"]
    .mean()
    .reindex(time_slots)
)

print("Training response rates:")
print((training_slot_rates * 100).round(2))

# 2. Choose the best slot using training data only.
best_baseline_slot = training_slot_rates.idxmax()

print("\nBest-time-slot baseline:", best_baseline_slot)

# 3. Evaluate notifications that actually occurred in that slot
# in the test data.
baseline_test_rows = test_original[
    test_original["time_slot"] == best_baseline_slot
]

if len(baseline_test_rows) > 0:
    baseline_test_rate = baseline_test_rows["responded"].mean()

    print("Test notifications in baseline slot:",
          len(baseline_test_rows))
    print("Observed test response rate:",
          round(baseline_test_rate * 100, 2), "%")
else:
    print("No test notifications were observed in this slot.")
print("original", round(train_original["responded"].mean() * 100, 2), "%")

Training response rates:
time_slot
early morning     3.92
mid morning      25.97
late morning     42.59
afternoon        42.62
evening          43.09
night            23.55
Name: responded, dtype: float64

Best-time-slot baseline: evening
Test notifications in baseline slot: 3051
Observed test response rate: 27.14 %
original 29.97 %
